In [3]:
from datasets import load_dataset

# 加载数据集
raw_datasets = load_dataset('glue', 'sst2')

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [21]:
# 查看数据集
raw_datasets['train'].features

{'sentence': Value('string'),
 'label': ClassLabel(names=['negative', 'positive']),
 'idx': Value('int32')}

In [10]:
from transformers import AutoTokenizer, DataCollatorWithPadding

# 从预训练模型获取tokenizer
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [28]:
# 输入tokenizer化助手函数
def tokenize_function(example):
    return tokenizer(example["sentence"], truncation=True)

In [29]:
# 准备好数据集和collate函数
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

In [13]:
# 查看准备好的数据抽样
tokenized_datasets['train'][0]

{'sentence': 'hide new secretions from the parental units ',
 'label': 0,
 'idx': 0,
 'input_ids': [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [14]:
from transformers import TrainingArguments

# 训练参数
training_args = TrainingArguments('test-trainer')

In [22]:
from transformers import AutoModelForSequenceClassification

# 预训练模型加载
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels = 2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [37]:
import evaluate
import numpy as np

# 度量函数
def compute_metrics(eval_preds):
    metric = evaluate.load('glue', 'sst2')
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions = predictions, references = labels)

In [38]:
from transformers import Trainer

# 训练器
trainer = Trainer(
    model,
    training_args,
    train_dataset = tokenized_datasets['train'],
    eval_dataset = tokenized_datasets['validation'],
    data_collator = data_collator,
    processing_class = tokenizer,
    compute_metrics = compute_metrics,
)

In [32]:
trainer.train()

Step,Training Loss
500,0.419200
1000,0.381773
1500,0.344098
2000,0.332703
2500,0.298323
3000,0.289600
3500,0.301966
4000,0.278994
4500,0.285961
5000,0.285850


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=25257, training_loss=0.18666461452658276, metrics={'train_runtime': 1002.1368, 'train_samples_per_second': 201.616, 'train_steps_per_second': 25.203, 'total_flos': 3082513027395900.0, 'train_loss': 0.18666461452658276, 'epoch': 3.0})

In [34]:
tokenized_datasets['validation']

Dataset({
    features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 872
})

In [42]:
predictions = trainer.predict(tokenized_datasets['validation'])

In [48]:
preds = predictions.predictions.argmax(axis=-1)

In [58]:
(preds == tokenized_datasets['validation']['label']).mean()

np.float64(0.8979357798165137)